In [1]:
print("hello")

hello


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
from langchain_openai import ChatOpenAI
llm= ChatOpenAI(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
    max_tokens=1000
)

In [4]:
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x0000018A2AE50350>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000018A2AE236B0>, root_client=<openai.OpenAI object at 0x0000018A2A5D27B0>, root_async_client=<openai.AsyncOpenAI object at 0x0000018A2A8FF830>, model_name='openai/gpt-oss-120b', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', openai_proxy=None, max_tokens=1000, stream_chunk_timeout=120.0)

In [5]:
# static prompt
prompts=[
    ("system","You are a python developer"),
    ("system","You are a Java developer"),
    ("user","How to sort an array")
]
response= llm.invoke(prompts)
print(response.content)

Below are several common ways to sort an array in Java.  
Choose the one that fits the type of data you’re working with and the ordering you need.

---

## 1️⃣ Using `Arrays.sort` for primitive types  

```java
import java.util.Arrays;

public class SortPrimitives {
    public static void main(String[] args) {
        int[] numbers = {5, 2, 9, 1, 7};

        // In‑place ascending sort
        Arrays.sort(numbers);

        System.out.println("Sorted: " + Arrays.toString(numbers));
        // Output: Sorted: [1, 2, 5, 7, 9]
    }
}
```

- Works for `int[]`, `double[]`, `char[]`, etc.  
- By default it sorts in **ascending** order.  

### Descending order (primitive)

```java
int[] numbers = {5, 2, 9, 1, 7};

// Box the primitives, sort with a comparator, then unbox
Integer[] boxed = Arrays.stream(numbers).boxed().toArray(Integer[]::new);
Arrays.sort(boxed, (a, b) -> b - a);   // descending

// If you need an int[] again:
int[] desc = Arrays.stream(boxed).mapToInt(Integer::intValue).toA

In [6]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

In [7]:
dynmaic_prompts=ChatPromptTemplate([
    {"role":"system","content":"you are a translator and translate the sentences to {language}"},
    {"role":"user","content":"{query}"}
])
dynmaic_prompts

ChatPromptTemplate(input_variables=['language', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='you are a translator and translate the sentences to {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

In [8]:
# chat_template=ChatPromptTemplate([
#     ('system',"you are a very helpful customer support agent"),
#     MessagesPlaceholder(variable_name="uploading_txt"),
#     ('human','{query}')
# ])
# chat_history=[]
# with open("../chat_history.txt",'r')as file:
#     chat_history.append(file.readlines())

# prompt=chat_template.invoke({
#     'uploading_txt':chat_history,
#     'query':"whats my order id"
# })
# res= llm.invoke(prompt)
# print(res)

In [9]:
from langchain_core.messages import HumanMessage, AIMessage 
chat_template= ChatPromptTemplate([ 
    ("system", "You are a helpful customer support agent."), 
                      MessagesPlaceholder(variable_name="chat_history"), ("human", "{query}") ]) 
chat_history = []
with open("../chat_history.txt", "r") as f: 
    for i, line in enumerate(f): 
        chat_history.append(line)
print(chat_history)

# with open("../chat_history.txt",'r') as file:
#     chat_history.extend(file.readlines())

prompts = chat_template.invoke({ 
    "chat_history": chat_history, 
    "query": "whats the customer order ID"
}) 
response = llm.invoke(prompts) 
print(response.content)

['Customer: Hi, I ordered a LakmÃ© foundation and Maybelline lipstick from Nykaa last week, but I received the wrong shade for the foundation.\n', 'Customer Care: Hello! Sorry for the inconvenience. Could you please share your order ID?\n', 'Customer: Sure, my order ID is NYK12345678.\n', 'Customer Care: Thank you. I checked your order and see that you ordered LakmÃ© 9to5 Primer + Matte Foundation in the shade Warm Beige.\n', 'Customer: Yes, but I received the shade Cool Ivory instead.\n', 'Customer Care: I understand. We can arrange a replacement or refund for the incorrect product. Which would you prefer?\n', 'Customer: I would like a replacement, please.\n', 'Customer Care: Sure! Your replacement request has been initiated. The correct product will be delivered within 5-7 business days.\n', 'Customer: Thank you. Also, the Maybelline lipstick packaging looked slightly damaged.\n', 'Customer Care: We apologize for that. If the product itself is damaged, you can upload images in the ap

# Structured OUTPUT

In [10]:
# typing module
from typing import TypedDict,Annotated, Optional,Literal
from pydantic import BaseModel,Field
# Schema
# class Review(TypedDict):
#     summary:Annotated[str,"A brief summary of the review"]
#     sentiment: Annotated[Literal["Pos","Neg"],"Return sentiment of the review"]
#     pros:Annotated[Optional[list[str]],"Write down all the pros"]


# schema using pydantic rules
class Review(BaseModel):
    summary:str=Field(description= "A brief summary of the review")
    sentiment: Literal["Pos","Neg"]=Field(description= "Return sentiment of the review")
    pros:Optional[list[str]]= Field(default=None,description="Write down all the pros")

sturtured_output= llm.with_structured_output(Review)

result= sturtured_output.invoke("""
I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Yamuna

""")
print(result)

summary='The Samsung Galaxy S24 Ultra delivers powerful performance, excellent camera, long battery life and useful S‑Pen support, though it’s heavy, has bloatware and a high price.' sentiment='Pos' pros=['Insanely powerful processor (great for gaming and productivity)', 'Stunning 200MP camera with incredible zoom capabilities', 'Long battery life with fast charging', 'S-Pen support is unique and useful']


In [ ]:
# chatbot
# langchain--- chain--- 1, 2, llm, translator, 4, 5 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt= PromptTemplate(
    template="Generate 5 interseting facts about {topic}",
    input_variables=['topic']
)
# Generate 5 interseting facts about {quantum physics}--- python language(transalators)(parser- its convert the row text to python)
parser= StrOutputParser()
chain= prompt|llm
result= chain.invoke({"topic":"football"})
print(result)
chain.get_graph().print_ascii()


content='**Five fascinating football (soccer) facts**\n\n1. **The game’s oldest surviving rulebook dates back to 1863** – The “Laws of the Game” were first codified by England’s newly formed Football Association. The original 13 rules still form the backbone of modern soccer, even though the book itself is now housed in the British Library.\n\n2. **A single World Cup final can out‑earn a small country’s GDP** – The 2018 FIFA World Cup final generated roughly **$1.2\u202fbillion** in global broadcast revenue alone. That’s more than the entire annual GDP of nations such as Bhutan, Belize, or the Maldives.\n\n3. **The fastest hat‑trick ever recorded came in under three minutes** – On March\u202f7,\u202f2004, Scottish striker **Rogério “Rogério” Ceni** (actually a goalkeeper‑turned‑outfield player) scored three goals for São Paulo FC in **2\u202fminutes\u202f46\u202fseconds** during a domestic cup match.\n\n4. **The most‑watched single sporting event in history isn’t the Super Bowl, it’s a

In [12]:
# langchain--- chain--- 1, 2, llm, translator, 4, 5 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt= PromptTemplate(
    template="Generate 5 interseting facts about {topic}",
    input_variables=['topic']
)
# Generate 5 interseting facts about {quantum physics}--- python language(transalators)(parser- its convert the row text to python)
parser= StrOutputParser()
chain= prompt|llm|parser
result= chain.invoke({"topic":"football"})
print(result)
chain.get_graph().print_ascii()

**Five fascinating football (soccer) facts**

1. **The World Cup’s staggering viewership**  
   The 2018 FIFA World Cup final attracted an estimated **1.12 billion viewers worldwide**, making it the most‑watched single sporting event in history at the time.

2. **A 2‑minute goal‑less streak**  
   In the 1995 Premier League match between Sunderland and Manchester United, the ball was in play for **2 minutes 57 seconds** without touching the ground—a record for the longest “air‑ball” run in top‑flight English football.

3. **The “Maracanã Miracle”**  
   In the 1950 World Cup final round, Uruguay’s 2‑1 victory over Brazil at the Maracanã Stadium shocked 200,000 fans, creating one of the most famous upsets and cementing the phrase “the Maracanã’s black day” in Brazilian lore.

4. **A single player’s 13‑second hat‑trick**  
   In 2004, **Sadio Mané** (then playing for FC Rennes) scored three goals in **13 seconds** during a Ligue 2 match, a feat that still ranks among the fastest hat‑tric

In [ ]:
p1= PromptTemplate(
    template='Generate short and simple notes from the  following text  \n{text}',
    input_variables=['text']
)
p2= PromptTemplate(
    template='Generate  5 short question answers from the following text  \n{text}',
    input_variables=['text']
)
p3= PromptTemplate(
    template='merge the provided notes and quiz into a single documenet  \n{text}',
    input_variables=['text']
)

parser=  StrOutputParser()

chain= p1 |llm| parser | p2 |llm | parser | p3 | llm | parser
result= chain.invoke({"text":"Unemployemnet  rate in World"})
print(result)

chain.get_graph().print_ascii()

**Global Unemployment Overview – Notes & Quiz**

---

### 1. Definition of the World Unemployment Rate  
The world unemployment rate is the **percentage of the global labor force that is without a job but is actively looking for work**. It is calculated as:  

\[
\text{Unemployment Rate} = \left(\frac{\text{Number of Unemployed}}{\text{Total Labor Force}}\right) \times 100
\]

---

### 2. Recent Global Average (2023‑2024)  
According to the International Labour Organization (ILO), the **average global unemployment rate for the 2023‑2024 period was roughly 5 %–6 %**.

---

### 3. Regions with the Highest Typical Unemployment Rates  
- **Developing and fragile states** tend to have the highest rates, often **10 % or higher**.  
- Illustrative examples:  
  - **South Africa:** ≈ 33 %  
  - **Certain Sub‑Saharan African nations:** ≈ 15 %

---

### 4. Impact of the COVID‑19 Pandemic  
- **2020:** Global unemployment surged to about **6.5 %** as economies shut down.  
- **2021‑2023:** The ra

In [ ]:
# parallel chains
from langchain_core.runnables import RunnableParallel
parallel_chain=  RunnableParallel({
    "notes":p1|llm|parser,
    "quiz":p2|llm|parser
})